# Import a Triangle Mesh Geoscience Object from an OBJ file

This notebook shows how to load and import a Triangle Mesh Geoscience Object from an OBJ file.

In the first cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In Cell 2, replace `"your-client-id"` with your Evo app client ID before running the cell.

You can then pick from the OBJ files that are placed in the `data` subdirectory of where this notebook is and choose to upload it.

There are multiple importer implementations supported, the recommended one is the default "trimesh", but "tinyobj" can also be
used to process larger meshes.

In [1]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(client_id="your-client-id").login()

C:\Users\Maitri.Anvekar\AppData\Roaming\Python\Python310\site-packages\evo\notebooks\authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

In [2]:
from evo.data_converters.common import create_evo_object_service_and_data_client

object_service_client, data_client = create_evo_object_service_and_data_client(service_manager_widget=manager)

In [3]:
from glob import glob
from pathlib import Path

import ipywidgets as widgets

data_dir = Path.cwd() / "data"
obj_files = [p for p in glob("**/*.obj", recursive=True, root_dir=data_dir)]

file_selector = widgets.RadioButtons(options=obj_files, disabled=False, layout={"width": "max-content"})

display(file_selector)

RadioButtons(layout=Layout(width='max-content'), options=('cube.obj', 'output\\object.obj'), value='cube.obj')

In [4]:
from evo.data_converters.obj.importer.obj_to_evo import convert_obj
from pprint import pp

test_file = data_dir / file_selector.value
assert test_file.exists()

tags = {"TagName": "Tag value"}
implementation = "trimesh"
objects_metadata = await convert_obj(
    filepath=test_file,
    implementation=implementation,
    epsg_code=None,
    service_manager_widget=manager,
    tags=tags,
    upload_path=f"test-obj-{implementation}",
    publish_objects=True,
    overwrite_existing_objects=True,
)

print()
print("These objects have now been published:")

for metadata in objects_metadata:
    pp(metadata, indent=4)


These objects have now been published:
ObjectMetadata(environment=Environment(hub_url='https://au.api.seequent.com',
                                       org_id=UUID('269e4988-1fa8-49fe-b68c-3a26c9541c7c'),
                                       workspace_id=UUID('1850bf93-ea96-4a91-945f-10aa73527d88')),
               id=UUID('81ca6d95-e43f-429e-8db8-488c01e492fd'),
               name='cube.obj.json',
               created_at=datetime.datetime(2026, 6, 5, 14, 54, 59, tzinfo=TzInfo(0)),
               created_by=ServiceUser(id=UUID('0f12f033-219c-47f0-adf7-2c4bca5a048f'),
                                      name='Maitri Anvekar',
                                      email='Maitri.Anvekar@bentley.com'),
               parent='test-obj-trimesh',
               schema_id=ObjectSchema(root_classification='objects',
                                      sub_classification='triangle-mesh',
                                      version=SchemaVersion(major=2,
                          